# Training Pipeline for AMFinder Training

In [29]:
# Azure ML Libraries
from azure.ai.ml.entities import Environment, CommandJob
from azure.identity import InteractiveBrowserCredential
from azure.ai.ml import MLClient, Input, Output, command
from azure.ai.ml import dsl
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import load_component
from azure.ai.ml.entities._assets.environment import BuildContext, Environment
import requests
import mlflow
from azure.ai.ml.sweep import Choice, Normal, Uniform
from azure.ai.ml.sweep import TruncationSelectionPolicy

## Pipeline configuration

In [ ]:
# True if the pipeline should be subitted to aml
run_pipeline = True
allow_preprep_caching = False
create_with_schedule = False

# Relevant variables for AzureML
compute_target_name = "gpu-cluster-small"
model_directory = ''
experiment_name = "126_training_run"
schedule_name = f"{experiment_name}-schedule"

# ML Client
azure_config = {
    "subscription_id": "redacted",
    "resource_group": "redacted",
    "workspace_name": "redacted",
    "storage_account": "redacted"
    }

subscription_id = azure_config["subscription_id"]
resource_group = azure_config["resource_group"]
workspace_name = azure_config["workspace_name"]
storage_account = azure_config["storage_account"]

credential=InteractiveBrowserCredential()

ml_client = MLClient(credential=credential,
                     subscription_id=subscription_id,
                     resource_group_name=resource_group,
                     workspace_name=workspace_name)

# Check if the compute compute resource is available
compute_resource = ml_client.compute.get(compute_target_name)

# ML Flow config
azureml_tracking_uri = ml_client.workspaces.get(
    ml_client.workspace_name
).mlflow_tracking_uri

azure_ml_mlflow_base = azureml_tracking_uri.replace("azureml://", "https://")
mlflow.set_tracking_uri(azure_ml_mlflow_base)

#deployment enviroment: - set default always to "qa"
#env = ml_client.environments.get(name="amf_aml_test_env", version="3")
#env = Environment()

#Potentially switch to definition on the fly by including the requirements_aml.txt right in the variable

# Select the enviroment for the python script steps
# pipeline_job_env = Environment(
#     image="mcr.microsoft.com/azureml/minimal-ubuntu22.04-py39-cuda11.8-gpu-inference:latest",
#     conda_file="./requirements_aml.yml"


## Default Paths

In [ ]:
# All registered data sources can be accessed through the ml client.
# It is also possible to add new data, like the temporary ones created below.
ml_ds_id = ml_client.datastores.get(storage_account).id
base_path = f"azureml:/{ml_ds_id}/paths/raw".replace("providers/Microsoft.MachineLearningServices/", "")


# Config base paths
small_subsample_dataset = base_path + r'/252_csv_Training_250312_6class/'
output_dir_path = base_path + r'/test_outdir/'

## Environment

In [ ]:
pipeline_job_env = ml_client.environments.get(name="amf_env_2", version="24") #Building the Environment in the UI allows for a specific Dockerfile to define commands, while using a conda yaml for further libraries.

# The following allows building an Environment from PythonSDK remotely.
# pipeline_job_env = Environment(
#     build=BuildContext(path='./environment_files/',             #if BuildContext is used, a conda filepath is not allowed. If a yaml with conda dependencies is ought to be used, an image needs to be specified first. In this case we need apt-get libvips however.
#     dockerfile_path = 'Dockerfile'                              #path specifies the folder which shall be uploaded in the context, whereas Dockerfile is the specific file for the customised Docker image.
#     ),
#     name="amf_train_env",
#     version="20",
#     description="training_environment",
#    )

# pipeline_job_env = Environment(
#     image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
#     name="amf_train_env",
#     version="8",
#     conda_file="./environment_files/requirements_aml.yml",
#     description="training_environment",
#    )

#pipeline_job_env.validate()

## Pipeline step definition

In [33]:
pipeline_step_AMFinder_126_training = command(
    name="amfinder126 training",
    display_name="",
    description="",
    code="../amf_code",                                      #Thats the code uploaded as a "repo" to Azure
    inputs={
        "images": Input(type="uri_folder",
                         path=small_subsample_dataset),                  #specify uri_folder as we are working with a cloud folder in fact
        "batch_size": 32,
        "learning_rate": 0.001,
        "adam_beta1": 0.9,
        "adam_beta2": 0.999,
        "balance_factor": 1.0,
    },
    outputs={
        "training_outputs": Output(type="uri_folder"),
    },
    command="""
            conda run -n requirements_aml python amf train \
            --use-csvs \
            --images ${{inputs.images}}\
            --outdir ${{outputs.training_outputs}}\
            --mlflow \
            --batch_size ${{inputs.batch_size}}\
            --learning_rate ${{inputs.learning_rate}}\
            --adam_beta1 ${{inputs.adam_beta1}}\
            --adam_beta2 ${{inputs.adam_beta2}}\
            --balance_factor ${{inputs.balance_factor}}\
            -a \
            """,
    environment=pipeline_job_env
)

In [34]:
# Override your inputs with parameter expressions
command_job_for_sweep = pipeline_step_AMFinder_126_training(
    batch_size=Choice(values=[32, 64]),
    learning_rate=Uniform(min_value=0.000001, max_value=0.00001),
    adam_beta1=Uniform(min_value=0.875, max_value=0.925),
    adam_beta2=Uniform(min_value=0.95, max_value=0.999),
    balance_factor=Uniform(min_value=0.75, max_value=1.25),
)

In [35]:
# Call sweep() on your command job to sweep over your parameter expressions
sweep_job = command_job_for_sweep.sweep(
    compute=compute_target_name,
    sampling_algorithm="random",
    primary_metric="Validation loss",
    goal="Minimize",
)

In [36]:
# Specify your experiment details
sweep_job.display_name = "126-amfinder-sweep"
sweep_job.experiment_name = "126-amfinder-sweeps"
sweep_job.description = "First trial of 126 AMFinder Sweep jobs"

# Define the limits for this sweep
sweep_job.set_limits(max_total_trials=50)

# Termination policy
sweep_job.early_termination = TruncationSelectionPolicy(
    evaluation_interval=5,
    truncation_percentage=30,
    delay_evaluation=10,
    )

In [37]:
# submit the sweep
returned_sweep_job = ml_client.create_or_update(sweep_job)
# get a URL for the status of the job
returned_sweep_job.services["Studio"].endpoint